# Integrate Valuations with Matched Players

This notebook merges player valuation data with the matched players table using `player_id` and season dates.

## 1. Import Dependencies

In [ ]:
import pandas as pd

## 2. Read the Datasets

In [ ]:
valuations = pd.read_csv('../datasets/Football Data from Transfermarkt/player_valuations.csv')
matched_players = pd.read_csv('matched_players_full.csv')
print(f"Valuations records: {len(valuations)}")
print(f"Matched players records: {len(matched_players)}")

## 3. Parse and Extract Season Dates

Handle both traditional mid-to-mid year seasons and calendar year seasons

In [ ]:
valuations['date'] = pd.to_datetime(valuations['date'])

def get_season_start_end(season_str):
    # Handles formats like '2022/2023', '2024', etc.
    if '/' in str(season_str):
        start, end = season_str.split('/')
        start_year = int(start)
        end_year = int(end)
        return pd.Timestamp(f'{start_year}-07-01'), pd.Timestamp(f'{end_year}-06-30')
    elif str(season_str).isdigit():
        year = int(season_str)
        return pd.Timestamp(f'{year}-01-01'), pd.Timestamp(f'{year}-12-31')
    else:
        return pd.NaT, pd.NaT

matched_players[['season_start', 'season_end']] = matched_players['Seasons'].apply(
    lambda x: pd.Series(get_season_start_end(x))
)
# pd.Series inside apply stores Timestamps as nanosecond integers in object columns — cast explicitly
matched_players['season_start'] = pd.to_datetime(matched_players['season_start'])
matched_players['season_end'] = pd.to_datetime(matched_players['season_end'])


## 4. Merge and filter valuations within season

In [ ]:
merged = pd.merge(
    matched_players,
    valuations,
    on='player_id',
    how='left',
    suffixes=('', '_valuation')
)

in_season = merged[
    (merged['date'] >= merged['season_start']) &
    (merged['date'] <= merged['season_end'])
].copy()

print(f"Total matched records: {len(merged)}")
print(f"Records with valuations in season: {len(in_season)}")


## 5. For each player and season, keep the latest valuation

In [ ]:
in_season.sort_values(['player_id', 'season_start', 'date'], inplace=True)
latest_valuation = in_season.groupby(['player_id', 'season_start', 'season_end'], as_index=False).last()
print(f"Total records after selecting latest valuation: {len(latest_valuation)}")


## 6. Save the integrated table

In [ ]:
# Select only the required columns
columns_to_save = ['Players', 'player_id', 'Seasons', 'season_start', 'season_end', 'Matches', 'Goals', 'Assists', 'Seasons Ratings', 'market_value_in_eur', 'date', 'Teams']
latest_valuation = latest_valuation[columns_to_save]

latest_valuation.to_csv('matched_players_with_valuations.csv', index=False)
print("Integrated table saved as 'matched_players_with_valuations.csv'")

## 7. Show a sample

In [ ]:
latest_valuation.head()